In [ ]:
import pandas as pd
import re
from pathlib import Path

# Load combined TSV
df = pd.read_csv('../outputs/carolinian_combined.tsv', sep='\t', dtype=str, keep_default_na=False)
print(f'Loaded {len(df)} rows from combined TSV')

# Stage 1 directory
STAGE1_DIR = Path('../outputs/stage-1')


## Check 1: Variant forms — extract from Stage 1 and cross-reference against TSV

In [ ]:
def extract_headword(text):
    """Extract the first bold headword from a line."""
    m = re.match(r'^<b>([^<]+)</b>', text.strip())
    return m.group(1).strip() if m else None

def extract_variants(text):
    """Extract variant forms from (or <i>variant1, variant2</i>) pattern."""
    m = re.search(r'\(or <i>([^<]+)</i>\)', text)
    if not m:
        return []
    raw = m.group(1)
    return [v.strip().rstrip('.') for v in raw.split(',') if v.strip()]

# Extract headword -> variants from all Stage 1 TSV files
ground_truth = {}  # headword -> {'variants': [...], 'page': str}

tsv_files = sorted(
    STAGE1_DIR.glob('*/page_*_stage1.tsv'),
    key=lambda p: int(''.join(filter(str.isdigit, p.parent.name)) or 0)
)

print(f'Found {len(tsv_files)} Stage 1 TSV files')

for tsv_path in tsv_files:
    page = tsv_path.parent.name
    stage1_df = pd.read_csv(tsv_path, sep='\t', dtype=str, keep_default_na=False)
    current_headword = None
    for _, row in stage1_df.iterrows():
        text = row.get('text', '').strip()
        if not text:
            continue
        hw = extract_headword(text)
        if hw:
            current_headword = hw
        if current_headword:
            variants = extract_variants(text)
            if variants:
                if current_headword not in ground_truth:
                    ground_truth[current_headword] = {'variants': variants, 'page': page}
                else:
                    ground_truth[current_headword]['variants'].extend(variants)

print(f'Found {len(ground_truth)} headwords with variant forms in Stage 1')


In [ ]:
# Build lookup: headword -> Variant_Form from combined TSV
tsv_variants = df.set_index('Headword')['Variant_Form'].to_dict()

issues = []

for headword, info in ground_truth.items():
    stage1_variants = set(info['variants'])
    tsv_variant_str = tsv_variants.get(headword, '')
    # Variant_Form may contain comma-separated values
    tsv_variant_set = set(v.strip() for v in tsv_variant_str.split(',') if v.strip())

    missing = stage1_variants - tsv_variant_set
    if missing:
        issues.append({
            'Page': info['page'],
            'Headword': headword,
            'Stage1_Variants': ', '.join(sorted(stage1_variants)),
            'TSV_Variant_Form': tsv_variant_str,
            'Missing_in_TSV': ', '.join(sorted(missing)),
        })

issues_df = pd.DataFrame(issues)
print(f'Flagged: {len(issues_df)} headwords with variant forms not fully captured in TSV')
if len(issues_df):
    display(issues_df)


## Check 2: TAN dialect headwords — extract from Stage 1 and cross-reference against TSV

In [ ]:
DIALECT_PATTERN = re.compile(r'^<b>([^<]+)</b>\s*\((TAN|EL|LN|S|tan|el|ln|s)\)', re.IGNORECASE)

def extract_dialect_headword(text):
    """Extract headword and dialect marker if line has <b>headword</b> (DIALECT)."""
    m = DIALECT_PATTERN.match(text.strip())
    if not m:
        return None, None
    return m.group(1).strip(), m.group(2).upper()

# Extract dialect-marked headwords from all Stage 1 TSV files
dialect_headwords = {}  # headword -> {'dialect': str, 'page': str}

for tsv_path in tsv_files:
    page = tsv_path.parent.name
    stage1_df = pd.read_csv(tsv_path, sep='\t', dtype=str, keep_default_na=False)
    for _, row in stage1_df.iterrows():
        text = row.get('text', '').strip()
        if not text:
            continue
        hw, dialect = extract_dialect_headword(text)
        if hw:
            dialect_headwords[hw] = {'dialect': dialect, 'page': page}

print(f'Found {len(dialect_headwords)} dialect-marked headwords in Stage 1')
by_dialect = {}
for hw, info in dialect_headwords.items():
    by_dialect.setdefault(info['dialect'], []).append(hw)
for d, hws in sorted(by_dialect.items()):
    print(f'  {d}: {len(hws)} headwords')


In [ ]:
# Cross-reference against combined TSV Dialect column
tsv_dialect = df.set_index('Headword')['Dialect'].to_dict()

issues = []
for headword, info in dialect_headwords.items():
    dialect_val = tsv_dialect.get(headword, '')
    if info['dialect'] not in dialect_val:
        issues.append({
            'Page': info['page'],
            'Headword': headword,
            'Expected_Dialect': info['dialect'],
            'TSV_Dialect': dialect_val if dialect_val else '(empty)',
        })

issues_df = pd.DataFrame(issues)
print(f'Flagged: {len(issues_df)} dialect-marked headwords not correctly captured in Dialect column')
if len(issues_df):
    display(issues_df)
else:
    print('All dialect-marked headwords correctly have their dialect in the Dialect column.')


## Check 3: Hallucination check — headwords vs English-Carolinian finderlist

In [ ]:
# Load finderlist
finderlist = pd.read_csv('../post-processing/finderlist.tsv', sep='\t', dtype=str, keep_default_na=False)
valid_headwords = set(finderlist['carolinian_headword'].str.strip().str.lower())
print(f'Loaded {len(valid_headwords)} unique Carolinian headwords from finderlist')


In [ ]:
MATCH_THRESHOLD = 0.70  # flag pages where fewer than 70% of headwords are in the finderlist

# Exclude variant/redirect entries from the check
is_variant = (
    df['Gloss'].str.strip().str.startswith('Variant of') |
    ((df['Variant_Form'].str.strip() != '') & (df['Gloss'].str.strip() == ''))
)
df_check = df[~is_variant].copy()

results = []
for page, group in df_check.groupby('Source_Page'):
    headwords = group['Headword'].str.strip().str.lower().tolist()
    total = len(headwords)
    matched = sum(1 for hw in headwords if hw in valid_headwords)
    match_rate = matched / total if total > 0 else 1.0
    results.append({
        'Source_Page': page,
        'Total_Headwords': total,
        'Matched': matched,
        'Unmatched': total - matched,
        'Match_Rate': round(match_rate, 2),
        'Flagged': match_rate < MATCH_THRESHOLD,
        'Unmatched_Headwords': ', '.join(hw for hw in group['Headword'].str.strip() if hw.lower() not in valid_headwords),
    })

results_df = pd.DataFrame(results).sort_values('Match_Rate')
flagged = results_df[results_df['Flagged']]
print(f'Flagged: {len(flagged)} pages with match rate below {MATCH_THRESHOLD:.0%}')
if len(flagged):
    display(flagged[['Source_Page', 'Total_Headwords', 'Matched', 'Unmatched', 'Match_Rate', 'Unmatched_Headwords']])
